In [ ]:
import joblib
import pandas as pd

# Path tới CSV cần test
CSV = 'csv/input.csv'

print('Test trên dữ liệu CSV: ', CSV)

# =============================================
# [1] LOAD CSV
# =============================================
print('[1] Loading CSV...')
df = pd.read_csv(CSV)
print(f'[2] Loaded {df.shape[0]} rows, {df.shape[1]} columns')

if df.shape[0] == 0:
    print('No data, aborting')
    exit(1)

# =============================================
# [2] LOAD TRAINED MODEL
# =============================================
print('[3] Loading model...')
rf = joblib.load('models/rf_model.pkl')  # model binary RF
# Model nhị phân, không cần LabelEncoder nếu đã là 0/1
# Nếu cần tên nhãn:
label_map = {0: 'BENIGN', 1: 'APT'}

# =============================================
# [3] DROP OBJECT COLUMNS KHÔNG CẦN
# =============================================
print('[4] Dropping object columns...')
object_cols = df.select_dtypes(include=['object']).columns.tolist()
df_model = df.drop(columns=object_cols, errors='ignore')
print(f'[4] After drop: {df_model.shape[1]} columns')

# =============================================
# [4] ALIGN FEATURES VỚI MODEL
# =============================================
print('[5] Aligning features with model...')
expected_cols = list(rf.feature_names_in_)

# Thêm các cột thiếu với giá trị 0
missing = [c for c in expected_cols if c not in df_model.columns]
if missing:
    print(f' - Adding missing columns: {missing}')
    for c in missing:
        df_model[c] = 0

# Chỉ giữ đúng thứ tự feature mà model train
X = df_model[expected_cols]
print(f'[6] Final feature matrix shape: {X.shape}')

# =============================================
# [5] PREDICTION
# =============================================
print('[7] Predicting...')
preds = rf.predict(X)
labels = [label_map[p] for p in preds]

# =============================================
# [6] SHOW SAMPLE PREDICTIONS
# =============================================
print('[8] Sample predictions: ', labels[:10])

# =============================================
# [7] COUNT APT
# =============================================
apt_count = sum(1 for l in labels if l != 'BENIGN')
print('[9] APT count:', apt_count)

# =============================================
# [8] LABEL DISTRIBUTION
# =============================================
print('\n[10] Label distribution:')
unique_labels = pd.Series(labels).value_counts()
for label, count in unique_labels.items():
    print(f' - {label}: {count}')
